# NB01 — The Overfitting Game
**Machine Learning in Finance · Week 1, Thursday**  [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ruiyuan-chen/ml-finance-course/blob/main/notebooks/NB01_overfitting_game.ipynb)

**Finance question:** Can you find a rule that predicts which stocks will beat the market?

**Key idea:** **In-sample vs out-of-sample.** A pattern found by searching past data may not hold in the future. The only honest test is on data you did not search.

In [ ]:
# ▶ RUN ME FIRST — loads the course tools and data (about 30 seconds the first time)
COURSE_URL = "https://raw.githubusercontent.com/ruiyuan-chen/ml-finance-course/main/"
import os, sys, urllib.request
here = [f for f in [".", "..", "../.."] if os.path.exists(os.path.join(f, "mlfin.py"))]
if here:
    sys.path.insert(0, here[0])
else:
    urllib.request.urlretrieve(COURSE_URL + "mlfin.py", "mlfin.py")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import mlfin
mlfin.setup(COURSE_URL)

## The game
On Tuesday you tried to write a rule that picked winning stocks by hand. Today the computer plays the
same game, but much faster: it invents **hundreds of random stock-picking rules**, checks how each
one would have done in the past, and shows us the best. Then we see what those "best" rules did
**afterwards**.

## Step 1: the stock table
`mlfin.make_features()` builds our main table: one row per stock per month (the 500 largest US
stocks), with facts about each stock that were known at that month-end (the **features**) and what
happened next month (`ret_next`, the return over the following month).

In [ ]:
panel = mlfin.make_features()
print(f"{len(panel):,} stock-months, {panel['ticker'].nunique()} different stocks")
panel.head()

What does each feature mean? (You'll meet them properly in Week 3.)

In [ ]:
pd.Series(mlfin.FEATURE_DESCRIPTIONS, name="meaning").to_frame()

## Step 2: invent random rules
Each rule combines a few random conditions, such as *"momentum in the top 30%"* (the stock is among
the 30% of stocks with the highest past-year return that month). These rules contain **no
intelligence at all** — they are dice rolls.

In [ ]:
# 🟡 YOUR TURN
N_RULES = 200             # ← how many random rules to try: try 20, then 1000
CONDITIONS_PER_RULE = 3   # ← conditions in each rule: try 1, then 5

rules = mlfin.make_random_rules(N_RULES, CONDITIONS_PER_RULE, seed=0)
for r in rules[:5]:
    print(r["rule_id"], ":", r["rule"])

## Step 3: search the past for the best rules
We score every rule over a **search period**: each month we "buy" the stocks that pass the rule and
compare their next-month return with SPY's. The score is the **average monthly return above SPY**
(in percent). 1% a month above the market would be an extraordinary, career-making result.

In [ ]:
# 🟡 YOUR TURN
SEARCH_START, SEARCH_END = "2018-01-31", "2020-12-31"   # ← the period we search in
LATER_START, LATER_END = "2021-01-31", "2023-12-31"     # ← the period that comes afterwards

In [ ]:
in_sample = mlfin.test_rules(panel, rules, SEARCH_START, SEARCH_END)
best = in_sample.sort_values("avg_monthly_vs_SPY_%", ascending=False).head(10)
best.round(2)

**Pause and discuss with your neighbour:** these rules beat the market by several percent *a
month* in the search period. Would you put your own money into the top rule? Why or why not?

## Step 4: what happened next?
Now the honest test: the **same rules**, scored on the later period that we did **not** search.

In [ ]:
later = mlfin.test_rules(panel, rules, LATER_START, LATER_END)
compare = pd.DataFrame({
    "search period (% a month vs SPY)": best["avg_monthly_vs_SPY_%"],
    "later period (% a month vs SPY)": later.loc[best.index, "avg_monthly_vs_SPY_%"],
    "rule": best["rule"]})
compare.round(2)

In [ ]:
print("Top-10 rules, search period:", round(compare.iloc[:, 0].mean(), 2), "% a month")
print("Top-10 rules, later period: ", round(compare.iloc[:, 1].mean(), 2), "% a month")
print("All 200 rules, later period:", round(later["avg_monthly_vs_SPY_%"].mean(), 2), "% a month")

## Step 5: the picture that explains the whole course

In [ ]:
corr = mlfin.plot_rules_in_vs_out(in_sample, later)
print(f"Correlation between search-period and later results: {corr:.2f}")

*What to look for:* if past success predicted future success, the dots would line up along a rising
diagonal. Instead they form a shapeless cloud, and the red "best" rules sit in the middle of it later on.
A correlation near 0 means the search-period score told us almost nothing about the future.

## The lesson: in-sample vs out-of-sample
* **In-sample** = the data we searched or trained on. Results there are always flattering, because we
  picked the winners *after* seeing the answers.
* **Out-of-sample** = data the search never saw. This is the only honest test.
* Try **200 rules** and some will look brilliant **by luck alone**. The more you try, the better the
  luckiest one looks — and the less it means.

Machine learning is a very fast rule-searcher. Everything in this course is about getting its benefits
without fooling ourselves like this.

## ⭐ GOING FURTHER (optional)
1. Change `N_RULES` to 1000 and re-run from Step 2. Does the best search-period score go up? Does the
   later score? Ask an AI assistant: *"Why does testing more rules make the best one look better by chance?"*
2. Change the periods to search 2010–2017 and test 2018–2025. With 8 years of data some rules keep part of
   their edge. Why might a longer search period be more trustworthy? (We'll come back to this in Week 12.)

## 🐞 Practice: fixing an error with an AI assistant
The next cell has a **deliberate mistake**. Run it and read the red error message. Then ask an AI
assistant for help the good way:
1. paste the **cell** and the **full error message**,
2. say what you **expected** it to do ("print the later-period average of the 10 best rules"),
3. ask it to **explain the cause**, not just give you new code.

Fix the cell and run it again.

In [ ]:
best_ids = in_sample["avg_monthly_vs_SPY_%"].nlargest(10).index
print("Best 10 rules in the later period:",
      round(later_results.loc[best_ids, "avg_monthly_vs_SPY_%"].mean(), 2), "% a month")

## 🪞 REFLECTION (graded — write in your own words, no AI)

Double-click this cell and replace the *...* with your answers.

1. **In one sentence, what finance question did this week's technique answer?**
   *...*
2. **What is one way the result could be misleading?**
   *...*
3. **What did you ask an AI assistant this week, and was its answer right?**
   *...*